In [1]:
!pip install transformers[torch] datasets accelerate scikit-learn

In [2]:
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score
from torch import nn

# Verify GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Working on: {device}")

Working on: cuda


In [3]:
train = pd.read_csv("train_model.csv")
test = pd.read_csv("test_model.csv")

In [4]:
train.head()

,text,rating
0,this place is terrible; the people in charge a...,2
1,terrible service! and they are saying that i n...,1
2,absolutely terrible company. they sent me to c...,1
3,"to find it, either park in front of the tuesda...",4
4,mall location. used their services for sedan. ...,4


In [5]:
test.head()

,text,rating
0,every time i’ve been hear they’ve done a great...,4
1,if u like calm and layed back without the ster...,4
2,i would give this place 0 if i could. my exper...,1
3,had some problems viewing a place and was not ...,3
4,the staff at this place is very rude! they do ...,1


In [6]:
train.isnull().sum()

,0
text,0
rating,0


In [7]:
test.isnull().sum()

,0
text,0
rating,0


In [8]:
train["rating"] = train["rating"] - 1

In [9]:
test["rating"] = test["rating"] - 1

In [10]:
# Calculate balanced weights
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train['rating']),
    y=train['rating']
)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(
    train[['text', 'rating']].rename(columns={'rating': 'labels'})
)

test_dataset = Dataset.from_pandas(
    test[['text', 'rating']].rename(columns={'rating': 'labels'})
)


In [11]:
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=160
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/288000 [00:00<?, ? examples/s]

Map:   0%|          | 0/72000 [00:00<?, ? examples/s]

In [13]:
class WeightedTrainer(Trainer):
    # We add 'num_items_in_batch=None' and '**kwargs' to match the new library requirements
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.get("labels")

        # Standard Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Our custom Weighted Loss logic (stays the same)
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="macro")
    }



In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
).to(device)

training_args = TrainingArguments(
    output_dir="./roberta_final",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.866375,0.847945,0.640792,0.595450
2,0.787770,0.838711,0.673736,0.617652


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=36000, training_loss=0.8437656589084201, metrics={'train_runtime': 5378.9359, 'train_samples_per_second': 107.084, 'train_steps_per_second': 6.693, 'total_flos': 4.736126564352e+16, 'train_loss': 0.8437656589084201, 'epoch': 2.0})

In [15]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

# Get predictions
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

# Per-class metrics
labels = ["1 star", "2 stars", "3 stars", "4 stars", "5 stars"]

print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=labels))

# Convert to DataFrame (nice table for report)
report = classification_report(y_true, y_pred, target_names=labels, output_dict=True)
report_df = pd.DataFrame(report).transpose()

print("\n=== Report as Table ===")
print(report_df)

# Confusion Matrix
print("\n=== Confusion Matrix ===")
cm = confusion_matrix(y_true, y_pred)
print(cm)

=== Classification Report ===
              precision    recall  f1-score   support

      1 star       0.88      0.78      0.83     27395
     2 stars       0.37      0.54      0.44      8055
     3 stars       0.44      0.48      0.46      8832
     4 stars       0.78      0.62      0.69     20518
     5 stars       0.57      0.81      0.67      7200

    accuracy                           0.67     72000
   macro avg       0.61      0.65      0.62     72000
weighted avg       0.71      0.67      0.68     72000


=== Report as Table ===
              precision    recall  f1-score       support
1 star         0.881889  0.779230  0.827387  27395.000000
2 stars        0.370114  0.538423  0.438679   8055.000000
3 stars        0.444925  0.477921  0.460833   8832.000000
4 stars        0.784523  0.621064  0.693289  20518.000000
5 stars        0.566499  0.814028  0.668072   7200.000000
accuracy       0.673736  0.673736  0.673736      0.673736
macro avg      0.609590  0.646133  0.617652  72000